In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras

print("Python ok")
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())

Python ok
TensorFlow: 2.21.0
Keras: 3.14.0
Backend: tensorflow


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# --- 1. CONFIGURAZIONE DEL BACKEND (Keras 3 è Multi-backend)
# Nel 2025 Keras è indipendente dal framework. Impostiamo TensorFlow come motore
# di calcolo per sfruttare le pipeline tf.data e il formato SavedModel. 
# In alternativa si può impostare "torch" per usare PyTorch come backend.
# Impostando "torch", chiediamo a Keras di tradurre i suoi layer in moduli PyTorch nativi.
# Va fatto PRIMA di importare keras perché la scelta del motore di calcolo è immutabile dopo il caricamento.
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras
from keras import layers, models
from keras.applications import ResNet50



In [3]:
# --- 2. ACQUISIZIONE E OTTIMIZZAZIONE DEL DATASET (tf_flowers) ---
# Scarichiamo il dataset ìtf_flowers' da TensorFlow Datasets. 
# Il dataset contiene 3.670 immagini di fiori suddivise in 5 classi.
dataset_url="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir=tf.keras.utils.get_file(origin=dataset_url,
                                 fname='flower_photos', untar=True)

# Caricamento con gestione automatica delle etichette e resize a 224x224 (standard ResNet)
# Utiliziamo un batch_size di 32, bilanciamento ideale tra velocità e stabilità del gradiente
train_ds,val_ds=keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# Ottimizzazione della pipeline I/O con caching e prefetching per ridurre i colli di bottiglia
# prefetching consente di preparare il batch successivo mentre il modello sta ancora addestrando sul batch corrente
train_ds=train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds=val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)


Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Using 734 files for validation.


In [ ]:
# --- 3. ARCHITETTURA DEL MODELLO CON AUGMENTATION INTEGRATA ---
# Definiamo la Data Augmentation come parte integrante del modello, 
# così da applicarla solo durante l'addestramento e non durante la validazione/test 
# (viene automaticamente disattivata durante validation e inferenza)
# Questo gatantisce che l'inferenza (TTA) avvenga esattamente con le stesse logiche del training.
img_augmentation=models.Sequential([
    layers.RandomFlip("horizontal"), # invarianza speculare
    layers.RandomRotation(0.1), # invarianza rotazionale
    layers.RandomZoom(0.1), # invarianza di scala (zoom in/out)
    layers.RandomContrast(0.1) # invarianza di contrasto
], name="augmentation_layer")

# Carichiamo ResNet50 pre-addestrato su ImageNet, escludendo il top layer (classificatore originale)
# quindi non parto da zero, parto da una rete che ha già visto 1.2 milioni di immagini di ImageNet
# rete che già riconosce bordi, curve, texture, occhi, zampe, ecc, dovo solo insegnarle quali di queste
# caratteristiche (bordi, curve, ecc) distinguono i 5 fiori
# include_top=false elimina l'ultimo classificatore, perchè quello riconosce le 1000 clssi di ImageNet
# che a me non servono, quindi non le carico
base_model=ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable=False # congeliamo i pesi del modello pre-addestrato per il transfer learning

# Costruzione del modello finale con Functional API
# e con Data Augmentation, ResNet50 e classificatore personalizzato
inputs=layers.Input(shape=(224, 224, 3))
x=img_augmentation(inputs)
x=keras.applications.resnet50.preprocess_input(x) # normalizzazione specifica di ResNet50
x=base_model(x, training=False) # passaggio attraverso ResNet50

# aggiunto la NUOVA TESTA (globalaverage, dropout, dense), che sarà l'unica parte effettivamente addestrata
x=layers.GlobalAveragePooling2D()(x) # pooling globale per ridurre la dimensionalità dell output per inserire poi un layer denso
x=layers.Dropout(0.5)(x) # dropout per ridurre l'overfitting
outputs=layers.Dense(5, activation='softmax')(x) # layer di output con softmax per 5 classi

model=models.Model(inputs, outputs, name="Flower_ResNet50_2025")


In [5]:
# --- 4. STRATEGIA DI ADDESTRAMENTO: WARM-UP + FINE-TUNING ---

# FASE 1:Addestramento rapido della 'testa' (Top Layers) con ResNet50 congelato
print("\n --- Fase 1: Warm-up del classificatore ---")
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=1, verbose=1)


 --- Fase 1: Warm-up del classificatore ---


92/92 ━━━━━━━━━━━━━━━━━━━━ 349s 4s/step - accuracy: 0.9775 - loss: 0.0758 - val_accuracy: 1.0000 - val_loss: 1.4638e-05


In [6]:
# FASE 2: Fine-tuning di ResNet50 con un learning rate molto basso per evitare di distruggere i pesi pre-addestrati
# sblocchiamo la base e ricongeliamo tutto tranne gli ultimi 10 layer (feature semantiche)
base_model.trainable=True
for layer in base_model.layers[:-10]:
    layer.trainable=False   

In [7]:
# Ricompilazione obbligatoria dopo il cambio di 'trainable'
# Usiamo un learning rate 100 volte più basso per non distruggere i pesi ImageNet
model.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
print("\n --- Fase 2: Fine-Tuning degli strati profondi ---")
model.fit(train_ds, validation_data=val_ds, epochs=5, verbose=1)



 --- Fase 2: Fine-Tuning degli strati profondi ---
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 617s 7s/step - accuracy: 1.0000 - loss: 7.0563e-05 - val_accuracy: 1.0000 - val_loss: 7.6126e-06
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 581s 6s/step - accuracy: 1.0000 - loss: 1.8460e-05 - val_accuracy: 1.0000 - val_loss: 2.9566e-06
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 602s 7s/step - accuracy: 1.0000 - loss: 1.0941e-05 - val_accuracy: 1.0000 - val_loss: 1.5438e-06
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 679s 7s/step - accuracy: 1.0000 - loss: 9.6966e-06 - val_accuracy: 1.0000 - val_loss: 9.9670e-07
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 556s 6s/step - accuracy: 1.0000 - loss: 6.7356e-06 - val_accuracy: 1.0000 - val_loss: 6.7335e-07


In [ ]:
# -- 5. TEST-TIME AUGMENTATION (TTA) 
# La TTA migliora la robustessa: facciamo 5 predizini diverse sulla stessa immagine
# (cambiando rotazione/flip casualmente) e ne facciamo la media
# eseguiamo la nostra predizione 5 volte per ogni immagine, perchè all'inizio del modello abbiamo un layer 
# di augmentation che cambia casualmente l'immagine. 
# In questo modo otteniamo 5 predizioni diverse per la stessa immagine e ne facciamo la media per ottenere una predizione più robusta.
def predict_with_tta(model,dataset,steps=5):
    print(f"Esecuzione TTA con {steps} passaggi...")
    tta_preds=[]
    for i in range(steps):
        preds=model.predict(dataset, verbose=0)
        tta_preds.append(preds)
    return np.mean(tta_preds,axis=0)


In [ ]:
# --- 6. ESPORTAZIONE E BENCHMARK PRESTAZIONALE ---
model_path="flower_resnet50_saved_model"
model.export(model_path)

# Caricamento del modello esportato per test di velocità (Interference Benchmarks)
reloaded_model=tf.saved_model.load(model_path)
infer=reloaded_model.signatures["serving_default"]

# Misurazione della latenza media su un batch
for images, labels in val_ds.take(1):
    start_time=time.time()
    # Esecuzione dell'inferenza tramite la firma di default del modello salvato
    # Nota: export mappa l'input sul nome del primo layer o "input_layer" se non specificato. 
    input_name=list(infer.structured_input_signature[1].keys())[0]
    infer(**{input_name: tf.constant(images)})
    end_time=time.time()

# Calcolo metriche di istanze e throughout
total_time=end_time-start_time
batch_size=32
latency_per_img=(total_time/batch_size)*1000 # in millisecondi
throughput=batch_size/total_time # immagini al secondo
print(f"\n --- PERFORMANCE REPORT (Post export) ---")
print(f"Latenza per immagine: {latency_per_img:.2f} ms")
print(f"Throughput: {throughput:.2f} immagini al secondo")
